In [22]:
############################################
## ユーザーが変更すべきハイパーパラメータ
############################################

## 1) ベースパス（outlier の直下まで）
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"

## 2) 解析対象の目的変数と、そのモデル名・dataset_tag の対応
target_config <- list(
  PCEmax = list(model_name = "SVM_Radial", folder_name = "PCE", dataset_tag = "m_all_OH"),
  Jsc    = list(model_name = "gprLinear",  folder_name = "Jsc", dataset_tag = "m_all_OH"),
  Voc    = list(model_name = "RF",         folder_name = "Voc", dataset_tag = "m_base_FP"),
  FF     = list(model_name = "kNN",        folder_name = "FF",  dataset_tag = "m_all_FP")
)

## 3) bundle フォルダ名
bundle_folder_name <- "bundle_for_thesis_20260119"

## 4) 誤差ファイル内の列名
col_true   <- "Observed"   # 実測値
col_pred   <- "Predicted"  # 予測値
col_abserr <- "AbsError"   # 絶対誤差
col_id     <- "SampleID"   # サンプルID

## 5) 外れ値抽出モード
use_top_k_mode   <- FALSE   # top_k は使わない
use_abs_thr_mode <- TRUE    # 固定しきい値を使う

## 6) top_k（保険として残しておくが使わない前提）
top_k <- 10

## 7) 目的変数ごとの固定 AbsError 閾値（必要に応じて調整）
abs_error_thr_list <- list(
  PCEmax = 3.5,
  Jsc    = 5.0,
  Voc    = 0.15,
  FF     = 0.20
)

## 共通の abs_error_thr（使わないなら NA のままでOK）
abs_error_thr <- NA_real_

## 8) 出力フォルダ名プレフィックス
output_prefix <- "outlier_results"

## 9) 総括フォルダ名
summary_folder_name <- "outlier_summary_fixedThr_20260119"


############################################
## ここから下は通常変更不要
############################################

suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(readr)
})

## 1目的変数を処理する関数
process_one_target <- function(target) {

  cfg <- target_config[[target]]
  if (is.null(cfg)) {
    cat("  [WARN] target_config に設定がありません: ", target, "\n")
    return(invisible(NULL))
  }

  model_name   <- cfg$model_name
  dataset_tag  <- cfg$dataset_tag
  folder_name  <- cfg$folder_name   # PCE, Jsc, Voc, FF

  cat("============================================\n")
  cat("Target:", target, "\n")
  cat(" model_name  :", model_name,  "\n")
  cat(" dataset_tag :", dataset_tag, "\n")
  cat(" folder_name :", folder_name, "\n")
  cat("============================================\n")

  ## フォルダとファイルパス生成
  bundle_path <- file.path(base_path, bundle_folder_name, folder_name)

  ## ファイル名のベース
  base_file_stub <- paste0(dataset_tag, "_rebuilt_", target)

  file_oof_err  <- file.path(bundle_path,
                             paste0(base_file_stub, "_pred_OOF_withError.csv"))
  file_oof_feat <- file.path(bundle_path,
                             paste0(base_file_stub, "_pred_OOF_withError_andFeatures.csv"))

  ## 出力フォルダ（目的変数ごと）
  out_dir <- file.path(base_path, paste0(output_prefix, "_", target))
  if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

  cat("Input (OOF+Error):     ", file_oof_err,  "\n")
  cat("Input (WithFeatures):  ", file_oof_feat, "\n")
  cat("Output dir:            ", out_dir,       "\n\n")

  if (!file.exists(file_oof_err)) {
    cat("  [WARN] OOF+Error ファイルが見つかりません。パスやファイル名を確認してください。\n\n")
    return(invisible(NULL))
  }
  if (!file.exists(file_oof_feat)) {
    cat("  [WARN] WithError_andFeatures ファイルが見つかりません。パスやファイル名を確認してください。\n\n")
    return(invisible(NULL))
  }

  oof_err <- read_csv(file_oof_err, show_col_types = FALSE)
  feat    <- read_csv(file_oof_feat, show_col_types = FALSE) %>%
    dplyr::select(-dplyr::any_of(c("Observed", "Predicted", "AbsError")))

  needed_cols_err <- c(col_true, col_pred, col_abserr, col_id)
  miss_err <- setdiff(needed_cols_err, colnames(oof_err))
  if (length(miss_err) > 0) {
    cat("  [ERROR] OOF+Error ファイルに必要な列がありません:\n")
    print(miss_err)
    return(invisible(NULL))
  }
  if (!(col_id %in% colnames(feat))) {
    cat("  [ERROR] Features ファイルに ID 列がありません: ", col_id, "\n")
    return(invisible(NULL))
  }

  ## 誤差概要をコンソールに表示
  cat("誤差 (", col_abserr, ") summary:\n", sep = "")
  print(summary(oof_err[[col_abserr]]))
  cat("\n")

  ## ヒストグラムを保存
  p_hist <- ggplot(oof_err, aes(x = .data[[col_abserr]])) +
    geom_histogram(bins = 30, color = "black", fill = "skyblue") +
    theme_bw() +
    labs(title = paste0("Histogram of abs_error (", target, ")"))
  ggsave(filename = file.path(out_dir, paste0("hist_abs_error_", target, ".png")),
         plot = p_hist, width = 6, height = 4)

  ## 閾値決定：目的変数ごとの固定値を最優先
  chosen_abs_thr <- NA_real_

  if (isTRUE(use_abs_thr_mode)) {
    if (!is.null(abs_error_thr_list[[target]])) {
      ## 1) target固有の固定しきい値
      chosen_abs_thr <- abs_error_thr_list[[target]]
      cat("abs_error_thr_list に基づき、",
          target, " の閾値として ", chosen_abs_thr, " を使用します。\n", sep = "")
    } else if (!is.na(abs_error_thr)) {
      ## 2) 共通しきい値
      chosen_abs_thr <- abs_error_thr
      cat("共通 abs_error_thr として指定された ",
          chosen_abs_thr, " を使用します。\n", sep = "")
    } else if (isTRUE(use_top_k_mode)) {
      ## 3) 明示的な値が無い場合のみ top_k を保険的に使用
      if (is.na(top_k) || top_k <= 0) {
        cat("  [WARN] top_k が正しく設定されていません。top_k モードは無効化します。\n")
      } else {
        oof_sorted <- oof_err %>% arrange(desc(.data[[col_abserr]]))
        k_eff <- min(top_k, nrow(oof_sorted))
        chosen_abs_thr <- oof_sorted[[col_abserr]][k_eff]
        cat("top_k モードにより、", target,
            " の閾値として ", chosen_abs_thr,
            " (上位 ", k_eff, " 件目) を使用します。\n", sep = "")
      }
    } else {
      ## 4) それでも決まらない場合は IQR ベース
      q   <- quantile(oof_err[[col_abserr]], probs = 0.75, na.rm = TRUE)
      iqr <- IQR(oof_err[[col_abserr]], na.rm = TRUE)
      chosen_abs_thr <- as.numeric(q + 2 * iqr)
      cat("abs_error_thr 未指定のため、IQR ベース閾値 ",
          chosen_abs_thr, " を使用します。\n", sep = "")
    }
  }

  cat("\n最終的に使用する abs_error 閾値 =", chosen_abs_thr, "\n\n")

  ## S フラグ付け
  oof_err <- oof_err %>%
    mutate(is_S = .data[[col_abserr]] >= chosen_abs_thr)

  n_S <- sum(oof_err$is_S, na.rm = TRUE)
  cat("S (外れ値候補) の件数:", n_S, "/", nrow(oof_err), "\n")
  if (n_S > 0) {
    cat("S の abs_error 範囲:\n")
    print(range(oof_err[[col_abserr]][oof_err$is_S], na.rm = TRUE))
  }
  cat("\n")

  ## 特徴量と結合
  dat_all <- feat %>%
    left_join(oof_err %>%
                select(.data[[col_id]], .data[[col_true]], .data[[col_pred]],
                       .data[[col_abserr]], is_S),
              by = col_id)

  cat("結合後データ行数:", nrow(dat_all), "\n\n")

  ## 出力ファイルパス
  file_S        <- file.path(out_dir, paste0("high_error_Samples_", target, ".csv"))
  file_all      <- file.path(out_dir, paste0("all_with_Sflag_", target, ".csv"))
  file_summ_S   <- file.path(out_dir, paste0("summary_S_numeric_", target, ".csv"))
  file_summ_nonS<- file.path(out_dir, paste0("summary_nonS_numeric_", target, ".csv"))

  ## CSV 出力
  dat_all %>% filter(is_S)  %>% write_csv(file_S)
  write_csv(dat_all, file_all)

  ## Obs vs Pred 散布図
  p_scatter <- ggplot(dat_all,
                      aes(x = .data[[col_true]],
                          y = .data[[col_pred]],
                          color = is_S)) +
    geom_point(alpha = 0.7) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
    theme_bw() +
    labs(title = paste0("Obs vs Pred (", target, ")"),
         x = paste0("Observed ", target),
         y = paste0("Predicted ", target))

  ggsave(filename = file.path(out_dir, paste0("obs_vs_pred_", target, ".png")),
         plot = p_scatter, width = 6, height = 5)

  ## S / 非S の数値要約
  num_cols <- sapply(dat_all, is.numeric)

  summ_S <- dat_all %>%
    filter(is_S) %>%
    summarise(across(which(num_cols), list(mean = mean, sd = sd), na.rm = TRUE))
  summ_nonS <- dat_all %>%
    filter(!is_S) %>%
    summarise(across(which(num_cols), list(mean = mean, sd = sd), na.rm = TRUE))

  write_csv(summ_S,    file_summ_S)
  write_csv(summ_nonS, file_summ_nonS)

  cat("結果ファイル出力完了:\n")
  cat("  S のみ:                  ", file_S, "\n")
  cat("  全サンプル+Sフラグ:      ", file_all, "\n")
  cat("  Obs vs Pred 図:          ", file.path(out_dir, paste0("obs_vs_pred_", target, ".png")), "\n")
  cat("  S 数値要約:              ", file_summ_S, "\n")
  cat("  非S 数値要約:            ", file_summ_nonS, "\n\n")

  invisible(NULL)
}

############################################
## メイン：4 目的変数を順に処理
############################################

cat("=========== 外れ値解析スクリプト開始 ===========\n\n")
cat("base_path:", base_path, "\n\n")

for (tg in names(target_config)) {
  process_one_target(tg)
}

cat("=========== 外れ値解析スクリプト終了 ===========\n\n")


############################################
## 総括フォルダを作って結果を一括コピー
############################################

summary_dir <- file.path(base_path, summary_folder_name)
if (!dir.exists(summary_dir)) dir.create(summary_dir, recursive = TRUE)

targets <- names(target_config)

for (tg in targets) {
  src_dir <- file.path(base_path, paste0(output_prefix, "_", tg))

  ## ターゲットごとのサブフォルダを総括フォルダ配下に作成
  tgt_dir <- file.path(summary_dir, tg)
  if (!dir.exists(tgt_dir)) dir.create(tgt_dir, recursive = TRUE)

  ## コピー対象のファイル名
  src_files <- c(
    paste0("high_error_Samples_",    tg, ".csv"),
    paste0("all_with_Sflag_",       tg, ".csv"),
    paste0("summary_S_numeric_",    tg, ".csv"),
    paste0("summary_nonS_numeric_", tg, ".csv"),
    paste0("obs_vs_pred_",          tg, ".png"),
    paste0("hist_abs_error_",       tg, ".png")
  )

  for (f in src_files) {
    src_path <- file.path(src_dir, f)
    if (file.exists(src_path)) {
      file.copy(src_path,
                to = file.path(tgt_dir, f),
                overwrite = TRUE)
    } else {
      message("[INFO] ファイルが見つかりませんでした: ", src_path)
    }
  }
}

cat("総括フォルダへのコピー完了:\n  ", summary_dir, "\n")


=========== <U+5916><U+308C><U+5024><U+89E3><U+6790><U+30B9><U+30AF><U+30EA><U+30D7><U+30C8><U+958B><U+59CB> ===========

base_path: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier 

Target: PCEmax 
 model_name  : SVM_Radial 
 dataset_tag : m_all_OH 
 folder_name : PCE 
Input (OOF+Error):      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/bundle_for_thesis_20260119/PCE/m_all_OH_rebuilt_PCEmax_pred_OOF_withError.csv 
Input (WithFeatures):   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/bundle_for_thesis_20260119/PCE/m_all_OH_rebuilt_PCEmax_pred_OOF_withError_andFeatures.csv 
Output dir:             /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax 

<U+8AA4><U+5DEE> (AbsError) summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.00063 0.33022 0.74673 0.95654 1.33754 5.47919 

abs_error_thr_list <U+306B><U+57FA><U+3065><U+304D><U+3001>PCEmax <U+306E><U+95BE><U+5024><U+3068><U+3

=========== <U+5916><U+308C><U+5024><U+89E3><U+6790><U+30B9><U+30AF><U+30EA><U+30D7><U+30C8><U+7D42><U+4E86> ===========

<U+7DCF><U+62EC><U+30D5><U+30A9><U+30EB><U+30C0><U+3078><U+306E><U+30B3><U+30D4><U+30FC><U+5B8C><U+4E86>:
   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_summary_fixedThr_20260119 


In [21]:
# ############################################
# ## ユーザーが変更すべきハイパーパラメータ
# ############################################

# ## 1) ベースパス（outlier の直下まで）
# ## 必須：自分の環境に合わせて変更
# base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"

# ## 2) 解析対象の目的変数と、そのモデル名・dataset_tag の対応
# ##   - PCEmax: SVM_Radial × m_all_OH.csv  → フォルダ: PCE
# ##   - Jsc   : gprLinear × m_all_OH.csv   → フォルダ: Jsc
# ##   - Voc   : RF        × m_base_FP.csv  → フォルダ: Voc
# ##   - FF    : kNN       × m_all_FP.csv   → フォルダ: FF
# target_config <- list(
#   PCEmax = list(model_name = "SVM_Radial", folder_name = "PCE", dataset_tag = "m_all_OH"),
#   Jsc    = list(model_name = "gprLinear",  folder_name = "Jsc", dataset_tag = "m_all_OH"),
#   Voc    = list(model_name = "RF",         folder_name = "Voc", dataset_tag = "m_base_FP"),
#   FF     = list(model_name = "kNN",        folder_name = "FF",  dataset_tag = "m_all_FP")
# )

# ## 3) bundle フォルダ名
# ##   outlier/bundle_for_thesis_20260119/<target_folder>/...
# bundle_folder_name <- "bundle_for_thesis_20260119"


# ## 4) 誤差ファイル内の列名
# ##   実データの列名に合わせて必要に応じて変更
# col_true   <- "Observed"   # 実測値
# col_pred   <- "Predicted"  # 予測値
# col_abserr <- "AbsError"   # 絶対誤差
# col_id     <- "SampleID"   # サンプルID


# ## 5) 外れ値抽出モード
# ##   両方 TRUE でもよいが、その場合「top_k ベースの閾値」が優先される
# use_top_k_mode   <- TRUE   # TRUE: 上位 k 件を S にする
# use_abs_thr_mode <- TRUE   # TRUE: 絶対誤差しきい値で S にする

# ## 6) 「上位 k 件」モードのパラメータ
# ##   use_top_k_mode = TRUE の場合は必須 (正の整数)。
# ##   各目的変数とも「数個〜十数個」になるよう、まず 10 から試すのがおすすめ。
# top_k <- 10

# ## 7) 「絶対誤差しきい値」モードのパラメータ
# ##   use_abs_thr_mode = TRUE のとき、指定すれば固定閾値として使われる。
# ##   NA のままなら「top_k から自動設定」になる。
# ##   PCEmax が 0〜1 スケールなら 0.05〜0.1 程度が目安。
# abs_error_thr <- NA_real_  # 必須ではない（NA も可）

# ## 8) 出力フォルダ名プレフィックス
# ##   base_path 直下に "<output_prefix>_<target>" フォルダを作成。
# output_prefix <- "outlier_results"

# ############################################
# ## ここから下は通常変更不要
# ############################################

# suppressPackageStartupMessages({
#   library(dplyr)
#   library(ggplot2)
#   library(readr)
# })

# ## 1目的変数を処理する関数
# process_one_target <- function(target) {

#   cfg <- target_config[[target]]
#   if (is.null(cfg)) {
#     cat("  [WARN] target_config に設定がありません: ", target, "\n")
#     return(invisible(NULL))
#   }

#   model_name   <- cfg$model_name
#   dataset_tag  <- cfg$dataset_tag
#   folder_name  <- cfg$folder_name   # PCE, Jsc, Voc, FF

#   cat("============================================\n")
#   cat("Target:", target, "\n")
#   cat(" model_name  :", model_name,  "\n")
#   cat(" dataset_tag :", dataset_tag, "\n")
#   cat(" folder_name :", folder_name, "\n")
#   cat("============================================\n")

#   ## フォルダとファイルパス生成
#   ## 例: .../outlier/bundle_for_thesis_20260119/PCE/
#   bundle_path <- file.path(base_path, bundle_folder_name, folder_name)

#   ## ファイル名のベース
#   ## 例: m_all_OH_rebuilt_PCEmax_pred_OOF_withError.csv
#   base_file_stub <- paste0(dataset_tag, "_rebuilt_", target)

#   file_oof_err  <- file.path(bundle_path,
#                              paste0(base_file_stub, "_pred_OOF_withError.csv"))
#   file_oof_feat <- file.path(bundle_path,
#                              paste0(base_file_stub, "_pred_OOF_withError_andFeatures.csv"))

#   ## 出力フォルダ（目的変数ごと）
#   out_dir <- file.path(base_path, paste0(output_prefix, "_", target))
#   if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

#   cat("Input (OOF+Error):     ", file_oof_err,  "\n")
#   cat("Input (WithFeatures):  ", file_oof_feat, "\n")
#   cat("Output dir:            ", out_dir,       "\n\n")

#   if (!file.exists(file_oof_err)) {
#     cat("  [WARN] OOF+Error ファイルが見つかりません。パスやファイル名を確認してください。\n\n")
#     return(invisible(NULL))
#   }
#   if (!file.exists(file_oof_feat)) {
#     cat("  [WARN] WithError_andFeatures ファイルが見つかりません。パスやファイル名を確認してください。\n\n")
#     return(invisible(NULL))
#   }

#     oof_err <- read_csv(file_oof_err, show_col_types = FALSE)

#     feat    <- read_csv(file_oof_feat, show_col_types = FALSE) %>%
#       dplyr::select(-dplyr::any_of(c("Observed", "Predicted", "AbsError")))

#   cat(">> oof_err cols:\n"); print(names(oof_err))
#   cat(">> feat cols:\n");    print(names(feat))

#   needed_cols_err <- c(col_true, col_pred, col_abserr, col_id)
#   miss_err <- setdiff(needed_cols_err, colnames(oof_err))
#   if (length(miss_err) > 0) {
#     cat("  [ERROR] OOF+Error ファイルに必要な列がありません:\n")
#     print(miss_err)
#     return(invisible(NULL))
#   }
#   if (!(col_id %in% colnames(feat))) {
#     cat("  [ERROR] Features ファイルに ID 列がありません: ", col_id, "\n")
#     return(invisible(NULL))
#   }

#   ## 誤差概要をコンソールに表示
#   cat("誤差 (", col_abserr, ") summary:\n", sep = "")
#   print(summary(oof_err[[col_abserr]]))
#   cat("\n")

#   ## ヒストグラムを保存
#   p_hist <- ggplot(oof_err, aes(x = .data[[col_abserr]])) +
#     geom_histogram(bins = 30, color = "black", fill = "skyblue") +
#     theme_bw() +
#     labs(title = paste0("Histogram of abs_error (", target, ")"))
#   ggsave(filename = file.path(out_dir, paste0("hist_abs_error_", target, ".png")),
#          plot = p_hist, width = 6, height = 4)

#   ## 閾値決定：top_k 優先
#   chosen_top_k  <- NA_integer_
#   thr_from_topk <- NA_real_

#   if (isTRUE(use_top_k_mode)) {
#     if (is.na(top_k) || top_k <= 0) {
#       cat("  [WARN] top_k が正しく設定されていません。top_k モードは無効化します。\n")
#     } else {
#       oof_sorted <- oof_err %>% arrange(desc(.data[[col_abserr]]))
#       k_eff <- min(top_k, nrow(oof_sorted))
#       thr_from_topk <- oof_sorted[[col_abserr]][k_eff]
#       chosen_top_k  <- k_eff

#       cat("top_k モード: 上位 ", k_eff, " 件目の abs_error = ", thr_from_topk,
#           " を閾値候補とします。\n", sep = "")
#     }
#   }

#   chosen_abs_thr <- abs_error_thr

#   if (isTRUE(use_abs_thr_mode)) {
#     if (is.na(abs_error_thr)) {
#       if (!is.na(chosen_top_k)) {
#         chosen_abs_thr <- thr_from_topk
#         cat("abs_error_thr 未指定のため、top_k に由来する ",
#             chosen_abs_thr, " を使用します。\n", sep = "")
#       } else {
#         ## 万一 top_k も使えない場合のバックアップ
#         q   <- quantile(oof_err[[col_abserr]], probs = 0.75, na.rm = TRUE)
#         iqr <- IQR(oof_err[[col_abserr]], na.rm = TRUE)
#         chosen_abs_thr <- as.numeric(q + 1.5 * iqr)
#         cat("abs_error_thr 未指定かつ top_k 無効のため、IQR ベース閾値 ",
#             chosen_abs_thr, " を使用します。\n", sep = "")
#       }
#     } else {
#       cat("abs_error_thr モード: 指定された閾値 ", chosen_abs_thr, " を使用します。\n", sep = "")
#     }
#   } else {
#     if (!is.na(chosen_top_k)) {
#       chosen_abs_thr <- thr_from_topk
#     } else {
#       chosen_abs_thr <- median(oof_err[[col_abserr]], na.rm = TRUE)
#     }
#   }

#   cat("\n最終的に使用する abs_error 閾値 =", chosen_abs_thr, "\n\n")

#   ## S フラグ付け
#   oof_err <- oof_err %>%
#     mutate(is_S = .data[[col_abserr]] >= chosen_abs_thr)

#   n_S <- sum(oof_err$is_S, na.rm = TRUE)
#   cat("S (外れ値候補) の件数:", n_S, "/", nrow(oof_err), "\n")
#   if (n_S > 0) {
#     cat("S の abs_error 範囲:\n")
#     print(range(oof_err[[col_abserr]][oof_err$is_S], na.rm = TRUE))
#   }
#   cat("\n")

#   dat_all <- feat %>%
#     left_join(oof_err %>%
#                 select(.data[[col_id]], .data[[col_true]], .data[[col_pred]],
#                        .data[[col_abserr]], is_S),
#               by = col_id)
#   cat(">> dat_all cols:\n"); print(names(dat_all))

#   cat("結合後データ行数:", nrow(dat_all), "\n\n")

#   ## 出力
#   file_S   <- file.path(out_dir, paste0("high_error_Samples_", target, ".csv"))
#   file_all <- file.path(out_dir, paste0("all_with_Sflag_", target, ".csv"))

#   dat_all %>% filter(is_S) %>% write_csv(file_S)
#   write_csv(dat_all, file_all)

#   p_scatter <- ggplot(dat_all,
#                       aes(x = .data[[col_true]],
#                           y = .data[[col_pred]],
#                           color = is_S)) +
#     geom_point(alpha = 0.7) +
#     geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
#     theme_bw() +
#     labs(title = paste0("Obs vs Pred (", target, ")"),
#          x = paste0("Observed ", target),
#          y = paste0("Predicted ", target))

#   ggsave(filename = file.path(out_dir, paste0("obs_vs_pred_", target, ".png")),
#          plot = p_scatter, width = 6, height = 5)

#   num_cols <- sapply(dat_all, is.numeric)

#   summ_S <- dat_all %>%
#     filter(is_S) %>%
#     summarise(across(which(num_cols), list(mean = mean, sd = sd), na.rm = TRUE))
#   summ_nonS <- dat_all %>%
#     filter(!is_S) %>%
#     summarise(across(which(num_cols), list(mean = mean, sd = sd), na.rm = TRUE))

#   file_summ_S    <- file.path(out_dir, paste0("summary_S_numeric_", target, ".csv"))
#   file_summ_nonS <- file.path(out_dir, paste0("summary_nonS_numeric_", target, ".csv"))

#   write_csv(summ_S,    file_summ_S)
#   write_csv(summ_nonS, file_summ_nonS)

#   cat("結果ファイル出力完了:\n")
#   cat("  S のみ:                  ", file_S, "\n")
#   cat("  全サンプル+Sフラグ:      ", file_all, "\n")
#   cat("  Obs vs Pred 図:          ", file.path(out_dir, paste0("obs_vs_pred_", target, ".png")), "\n")
#   cat("  S 数値要約:              ", file_summ_S, "\n")
#   cat("  非S 数値要約:            ", file_summ_nonS, "\n\n")

#   invisible(NULL)
# }

# ## メイン：4 目的変数を順に処理
# cat("=========== 外れ値解析スクリプト開始 ===========\n\n")
# cat("base_path:", base_path, "\n\n")

# for (tg in names(target_config)) {
#   process_one_target(tg)
# }

# cat("=========== 外れ値解析スクリプト終了 ===========\n")


=========== <U+5916><U+308C><U+5024><U+89E3><U+6790><U+30B9><U+30AF><U+30EA><U+30D7><U+30C8><U+958B><U+59CB> ===========

base_path: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier 

Target: PCEmax 
 model_name  : SVM_Radial 
 dataset_tag : m_all_OH 
 folder_name : PCE 
Input (OOF+Error):      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/bundle_for_thesis_20260119/PCE/m_all_OH_rebuilt_PCEmax_pred_OOF_withError.csv 
Input (WithFeatures):   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/bundle_for_thesis_20260119/PCE/m_all_OH_rebuilt_PCEmax_pred_OOF_withError_andFeatures.csv 
Output dir:             /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax 

>> oof_err cols:
[1] "SampleID"           "foldID"             "Observed"          
[4] "Predicted"          "Error"              "AbsError"          
[7] "Type"               "HighErrorFlag"      "HighErrorThreshold"
>> feat cols:
  [1] 

top_k <U+30E2><U+30FC><U+30C9>: <U+4E0A><U+4F4D> 10 <U+4EF6><U+76EE><U+306E> abs_error = 3.956352 <U+3092><U+95BE><U+5024><U+5019><U+88DC><U+3068><U+3057><U+307E><U+3059><U+3002>
abs_error_thr <U+672A><U+6307><U+5B9A><U+306E><U+305F><U+3081><U+3001>top_k <U+306B><U+7531><U+6765><U+3059><U+308B> 3.956352 <U+3092><U+4F7F><U+7528><U+3057><U+307E><U+3059><U+3002>

<U+6700><U+7D42><U+7684><U+306B><U+4F7F><U+7528><U+3059><U+308B> abs_error <U+95BE><U+5024> = 3.956352 

S (<U+5916><U+308C><U+5024><U+5019><U+88DC>) <U+306E><U+4EF6><U+6570>: 10 / 1045 
S <U+306E> abs_error <U+7BC4><U+56F2>:
[1] 3.956352 5.479190

>> dat_all cols:
  [1] "SampleID"                                                                                                      
  [2] "foldID"                                                                                                        
  [3] "Error"                                                                                                         
  [4] "Type"  

<U+7D50><U+679C><U+30D5><U+30A1><U+30A4><U+30EB><U+51FA><U+529B><U+5B8C><U+4E86>:
  S <U+306E><U+307F>:                   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax/high_error_Samples_PCEmax.csv 
  <U+5168><U+30B5><U+30F3><U+30D7><U+30EB>+S<U+30D5><U+30E9><U+30B0>:       /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax/all_with_Sflag_PCEmax.csv 
  Obs vs Pred <U+56F3>:           /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax/obs_vs_pred_PCEmax.png 
  S <U+6570><U+5024><U+8981><U+7D04>:               /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax/summary_S_numeric_PCEmax.csv 
  <U+975E>S <U+6570><U+5024><U+8981><U+7D04>:             /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax/summary_nonS_numeric_PCEmax.csv 

Target: Jsc 
 model_name  : gprLinear 
 dataset_tag : m_a

top_k <U+30E2><U+30FC><U+30C9>: <U+4E0A><U+4F4D> 10 <U+4EF6><U+76EE><U+306E> abs_error = 7.726259 <U+3092><U+95BE><U+5024><U+5019><U+88DC><U+3068><U+3057><U+307E><U+3059><U+3002>
abs_error_thr <U+672A><U+6307><U+5B9A><U+306E><U+305F><U+3081><U+3001>top_k <U+306B><U+7531><U+6765><U+3059><U+308B> 7.726259 <U+3092><U+4F7F><U+7528><U+3057><U+307E><U+3059><U+3002>

<U+6700><U+7D42><U+7684><U+306B><U+4F7F><U+7528><U+3059><U+308B> abs_error <U+95BE><U+5024> = 7.726259 

S (<U+5916><U+308C><U+5024><U+5019><U+88DC>) <U+306E><U+4EF6><U+6570>: 10 / 1045 
S <U+306E> abs_error <U+7BC4><U+56F2>:
[1]  7.726259 11.000708

>> dat_all cols:
  [1] "SampleID"                                                                                                      
  [2] "foldID"                                                                                                        
  [3] "Error"                                                                                                         
  [4] "Type"

<U+7D50><U+679C><U+30D5><U+30A1><U+30A4><U+30EB><U+51FA><U+529B><U+5B8C><U+4E86>:
  S <U+306E><U+307F>:                   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Jsc/high_error_Samples_Jsc.csv 
  <U+5168><U+30B5><U+30F3><U+30D7><U+30EB>+S<U+30D5><U+30E9><U+30B0>:       /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Jsc/all_with_Sflag_Jsc.csv 
  Obs vs Pred <U+56F3>:           /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Jsc/obs_vs_pred_Jsc.png 
  S <U+6570><U+5024><U+8981><U+7D04>:               /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Jsc/summary_S_numeric_Jsc.csv 
  <U+975E>S <U+6570><U+5024><U+8981><U+7D04>:             /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Jsc/summary_nonS_numeric_Jsc.csv 

Target: Voc 
 model_name  : RF 
 dataset_tag : m_base_FP 
 folder_name : Voc 
Input (OO

<U+7D50><U+679C><U+30D5><U+30A1><U+30A4><U+30EB><U+51FA><U+529B><U+5B8C><U+4E86>:
  S <U+306E><U+307F>:                   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Voc/high_error_Samples_Voc.csv 
  <U+5168><U+30B5><U+30F3><U+30D7><U+30EB>+S<U+30D5><U+30E9><U+30B0>:       /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Voc/all_with_Sflag_Voc.csv 
  Obs vs Pred <U+56F3>:           /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Voc/obs_vs_pred_Voc.png 
  S <U+6570><U+5024><U+8981><U+7D04>:               /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Voc/summary_S_numeric_Voc.csv 
  <U+975E>S <U+6570><U+5024><U+8981><U+7D04>:             /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_Voc/summary_nonS_numeric_Voc.csv 

Target: FF 
 model_name  : kNN 
 dataset_tag : m_all_FP 
 folder_name : FF 
Input (OOF+

top_k <U+30E2><U+30FC><U+30C9>: <U+4E0A><U+4F4D> 10 <U+4EF6><U+76EE><U+306E> abs_error = 0.2571667 <U+3092><U+95BE><U+5024><U+5019><U+88DC><U+3068><U+3057><U+307E><U+3059><U+3002>
abs_error_thr <U+672A><U+6307><U+5B9A><U+306E><U+305F><U+3081><U+3001>top_k <U+306B><U+7531><U+6765><U+3059><U+308B> 0.2571667 <U+3092><U+4F7F><U+7528><U+3057><U+307E><U+3059><U+3002>

<U+6700><U+7D42><U+7684><U+306B><U+4F7F><U+7528><U+3059><U+308B> abs_error <U+95BE><U+5024> = 0.2571667 

S (<U+5916><U+308C><U+5024><U+5019><U+88DC>) <U+306E><U+4EF6><U+6570>: 10 / 1045 
S <U+306E> abs_error <U+7BC4><U+56F2>:
[1] 0.2571667 0.3900000

>> dat_all cols:
  [1] "SampleID"                         "foldID"                          
  [3] "Error"                            "Type"                            
  [5] "HighErrorFlag"                    "HighErrorThreshold"              
  [7] "THFSVAafetrN"                     "THFSVAafetrI"                    
  [9] "Aarea"                            "Aonsetp1Mfilm"      

<U+7D50><U+679C><U+30D5><U+30A1><U+30A4><U+30EB><U+51FA><U+529B><U+5B8C><U+4E86>:
  S <U+306E><U+307F>:                   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_FF/high_error_Samples_FF.csv 
  <U+5168><U+30B5><U+30F3><U+30D7><U+30EB>+S<U+30D5><U+30E9><U+30B0>:       /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_FF/all_with_Sflag_FF.csv 
  Obs vs Pred <U+56F3>:           /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_FF/obs_vs_pred_FF.png 
  S <U+6570><U+5024><U+8981><U+7D04>:               /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_FF/summary_S_numeric_FF.csv 
  <U+975E>S <U+6570><U+5024><U+8981><U+7D04>:             /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_FF/summary_nonS_numeric_FF.csv 

=========== <U+5916><U+308C><U+5024><U+89E3><U+6790><U+30B9><U+30AF><U+30EA><U+30D7><U+30C8><U+7D

In [24]:
library(dplyr)
library(readr)
library(tidyr)

base_path       <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
output_prefix   <- "outlier_results"
target_config   <- list(
  PCEmax = list(folder_name = "PCE"),
  Jsc    = list(folder_name = "Jsc"),
  Voc    = list(folder_name = "Voc"),
  FF     = list(folder_name = "FF")
)
targets <- names(target_config)

# 関心のある特徴量名（実データの列名に合わせて調整）
key_features <- c(
  "EgDFTp1M",      # Eg
  "Lay2thickness", # 膜厚
  "Aarea",         # 素子面積など
  "TAtemp1",       # アニール温度
  "TAtime1"        # アニール時間
  # 必要に応じて追加
)
basic_list <- lapply(targets, function(tg) {
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))

  dat <- read_csv(file_all, show_col_types = FALSE)

  # しきい値 = S / 非S の境界（S の中の最小 AbsError）
  thr <- dat %>%
    filter(is_S) %>%
    summarise(thr = min(AbsError, na.rm = TRUE)) %>%
    pull(thr)

  # S / 非S の件数と比率
  n_tot <- nrow(dat)
  n_S   <- sum(dat$is_S, na.rm = TRUE)
  n_nonS<- n_tot - n_S

  tibble(
    target         = tg,
    thr_AbsError   = thr,
    n_total        = n_tot,
    n_S            = n_S,
    n_nonS         = n_nonS,
    rate_S         = n_S / n_tot
  )
})

basic_summary <- bind_rows(basic_list)
write_csv(basic_summary,
          file.path(base_path, "summary_basic_S_rate_by_target.csv"))
abs_summary_list <- lapply(targets, function(tg) {
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))

  dat <- read_csv(file_all, show_col_types = FALSE)

  dat %>%
    group_by(is_S) %>%
    summarise(
      n      = n(),
      min    = min(AbsError, na.rm = TRUE),
      q1     = quantile(AbsError, 0.25, na.rm = TRUE),
      median = median(AbsError, na.rm = TRUE),
      mean   = mean(AbsError, na.rm = TRUE),
      q3     = quantile(AbsError, 0.75, na.rm = TRUE),
      max    = max(AbsError, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    mutate(target = tg) %>%
    select(target, is_S, n, min, q1, median, mean, q3, max)
})

abs_summary <- bind_rows(abs_summary_list)
write_csv(abs_summary,
          file.path(base_path, "summary_AbsError_S_vs_nonS_by_target.csv"))
feat_summary_list <- lapply(targets, function(tg) {
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))

  dat <- read_csv(file_all, show_col_types = FALSE)

  # 実際に存在する列だけに絞る（ターゲットによっては無い特徴量もあるので）
  feat_exist <- intersect(key_features, colnames(dat))
  if (length(feat_exist) == 0) return(NULL)

  dat %>%
    group_by(is_S) %>%
    summarise(
      across(
        all_of(feat_exist),
        list(mean = ~mean(.x, na.rm = TRUE),
             sd   = ~sd(.x,   na.rm = TRUE)),
        .names = "{.col}_{.fn}"
      ),
      .groups = "drop"
    ) %>%
    mutate(target = tg) %>%
    select(target, is_S, everything())
})

feat_summary <- bind_rows(feat_summary_list)
write_csv(feat_summary,
          file.path(base_path, "summary_features_S_vs_nonS_by_target.csv"))
# 3-1 の表から差分を計算
feat_long <- feat_summary %>%
  pivot_longer(
    cols = -c(target, is_S),
    names_to  = "feat_stat",
    values_to = "value"
  ) %>%
  separate(feat_stat, into = c("feature", "stat"), sep = "_(?=[^_]+$)")

feat_diff <- feat_long %>%
  filter(stat == "mean") %>%
  select(target, is_S, feature, value) %>%
  pivot_wider(
    names_from  = is_S,
    values_from = value,
    names_prefix = "mean_is_S_"
  ) %>%
  mutate(
    diff_S_minus_nonS = mean_is_S_TRUE - mean_is_S_FALSE
  )

write_csv(feat_diff,
          file.path(base_path, "summary_feature_mean_diff_S_minus_nonS_by_target.csv"))
model_difficulty <- abs_summary %>%
  group_by(target, is_S) %>%
  summarise(
    median_AbsError = median,
    mean_AbsError   = mean,
    .groups = "drop"
  ) %>%
  mutate(group = ifelse(is_S, "S", "nonS")) %>%
  select(target, group, median_AbsError, mean_AbsError) %>%
  pivot_wider(
    names_from  = group,
    values_from = c(median_AbsError, mean_AbsError),
    names_sep   = "_"
  ) %>%
  left_join(basic_summary, by = "target")

write_csv(model_difficulty,
          file.path(base_path, "summary_model_difficulty_by_target.csv"))


ERROR: Error: object 'feat' not found


In [25]:
library(dplyr)
library(readr)

# しきい値などの設定 --------------------------
file_diff <- "summary_feature_mean_diff_S_minus_nonS_by_target.csv"

# 「重要そうな目的変数」
important_targets <- c("Jsc", "PCEmax")

# diff の絶対値しきい値
abs_diff_thr <- 0.05

# 何個以上の重要ターゲットで差が出ていれば採用するか
min_imp_targets <- 1   # 例: 少なくとも Jsc または PCEmax のどちらかで効いていればOK

# 何個以上の全ターゲットで同じ向きの差が出ていれば「一貫している」とみなすか
min_total_targets_same_sign <- 2


# データ読み込み -----------------------------
diff_tbl <- read_csv(file_diff, show_col_types = FALSE)
# 列: target, feature, mean_is_S_FALSE, mean_is_S_TRUE, diff_S_minus_nonS


# 1) まず diff の絶対値が大きい行だけに絞る --------------------------
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)


# 2) 重要ターゲットでの効き具合を集計 -------------------------------
imp_effect <- diff_large %>%
  filter(target %in% important_targets) %>%
  group_by(feature) %>%
  summarise(
    n_imp_targets = n(),                                 # Jsc / PCEmax で効いている数
    max_abs_diff_imp = max(abs_diff),                   # その中で一番大きい差
    .groups = "drop"
  ) %>%
  filter(n_imp_targets >= min_imp_targets)


# 3) 全ターゲットでの「符号の一貫性」を評価 -------------------------
sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets = n(),                                    # 有効なターゲット数
    n_pos     = sum(sign > 0),
    n_neg     = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups   = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),                   # 多い方の符号の個数
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )


# 4) 重要ターゲットで効いていて、かつ符号も一貫している変数だけ抽出 ----
suspicious_vars <- imp_effect %>%
  inner_join(sign_consistency, by = "feature") %>%
  arrange(desc(max_abs_diff_imp))

suspicious_vars


feature,n_imp_targets,max_abs_diff_imp,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>,<int>,<chr>
EgDFTp1M,2,0.11452591,2,0,2,0.11452591,2,negative
TAtime1,2,0.11394941,2,2,0,0.11394941,2,positive
Aarea,2,0.07347052,2,0,2,0.07347052,2,negative


In [26]:
library(dplyr)
library(readr)

# しきい値などの設定 --------------------------
file_diff <- "summary_feature_mean_diff_S_minus_nonS_by_target.csv"

# 「重要そうな目的変数」
important_targets <- c("PCEmax")

# diff の絶対値しきい値
abs_diff_thr <- 0.05

# 何個以上の重要ターゲットで差が出ていれば採用するか
min_imp_targets <- 1   # 例: 少なくとも Jsc または PCEmax のどちらかで効いていればOK

# 何個以上の全ターゲットで同じ向きの差が出ていれば「一貫している」とみなすか
min_total_targets_same_sign <- 2


# データ読み込み -----------------------------
diff_tbl <- read_csv(file_diff, show_col_types = FALSE)
# 列: target, feature, mean_is_S_FALSE, mean_is_S_TRUE, diff_S_minus_nonS


# 1) まず diff の絶対値が大きい行だけに絞る --------------------------
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)


# 2) 重要ターゲットでの効き具合を集計 -------------------------------
imp_effect <- diff_large %>%
  filter(target %in% important_targets) %>%
  group_by(feature) %>%
  summarise(
    n_imp_targets = n(),                                 # Jsc / PCEmax で効いている数
    max_abs_diff_imp = max(abs_diff),                   # その中で一番大きい差
    .groups = "drop"
  ) %>%
  filter(n_imp_targets >= min_imp_targets)


# 3) 全ターゲットでの「符号の一貫性」を評価 -------------------------
sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets = n(),                                    # 有効なターゲット数
    n_pos     = sum(sign > 0),
    n_neg     = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups   = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),                   # 多い方の符号の個数
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )


# 4) 重要ターゲットで効いていて、かつ符号も一貫している変数だけ抽出 ----
suspicious_vars <- imp_effect %>%
  inner_join(sign_consistency, by = "feature") %>%
  arrange(desc(max_abs_diff_imp))

suspicious_vars


feature,n_imp_targets,max_abs_diff_imp,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>,<int>,<chr>
TAtime1,1,0.09644862,2,2,0,0.11394941,2,positive
EgDFTp1M,1,0.07470064,2,0,2,0.11452591,2,negative
Aarea,1,0.07347052,2,0,2,0.07347052,2,negative


In [27]:
library(dplyr)
library(readr)

# しきい値などの設定 --------------------------
file_diff <- "summary_feature_mean_diff_S_minus_nonS_by_target.csv"

# 「重要そうな目的変数」
important_targets <- c("Jsc")

# diff の絶対値しきい値
abs_diff_thr <- 0.05

# 何個以上の重要ターゲットで差が出ていれば採用するか
min_imp_targets <- 1   # 例: 少なくとも Jsc または PCEmax のどちらかで効いていればOK

# 何個以上の全ターゲットで同じ向きの差が出ていれば「一貫している」とみなすか
min_total_targets_same_sign <- 2


# データ読み込み -----------------------------
diff_tbl <- read_csv(file_diff, show_col_types = FALSE)
# 列: target, feature, mean_is_S_FALSE, mean_is_S_TRUE, diff_S_minus_nonS


# 1) まず diff の絶対値が大きい行だけに絞る --------------------------
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)


# 2) 重要ターゲットでの効き具合を集計 -------------------------------
imp_effect <- diff_large %>%
  filter(target %in% important_targets) %>%
  group_by(feature) %>%
  summarise(
    n_imp_targets = n(),                                 # Jsc / PCEmax で効いている数
    max_abs_diff_imp = max(abs_diff),                   # その中で一番大きい差
    .groups = "drop"
  ) %>%
  filter(n_imp_targets >= min_imp_targets)


# 3) 全ターゲットでの「符号の一貫性」を評価 -------------------------
sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets = n(),                                    # 有効なターゲット数
    n_pos     = sum(sign > 0),
    n_neg     = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups   = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),                   # 多い方の符号の個数
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )


# 4) 重要ターゲットで効いていて、かつ符号も一貫している変数だけ抽出 ----
suspicious_vars <- imp_effect %>%
  inner_join(sign_consistency, by = "feature") %>%
  arrange(desc(max_abs_diff_imp))

suspicious_vars


feature,n_imp_targets,max_abs_diff_imp,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>,<int>,<chr>
EgDFTp1M,1,0.11452591,2,0,2,0.11452591,2,negative
TAtime1,1,0.11394941,2,2,0,0.11394941,2,positive
Aarea,1,0.06829025,2,0,2,0.07347052,2,negative


In [49]:
library(dplyr)
library(readr)

# しきい値などの設定 --------------------------
file_diff <- "summary_feature_mean_diff_S_minus_nonS_by_target.csv"

# 「重要そうな目的変数」
important_targets <- c("Voc")

# diff の絶対値しきい値
abs_diff_thr <- 0.000001

# 何個以上の重要ターゲットで差が出ていれば採用するか
min_imp_targets <- 1   # 例: 少なくとも Jsc または PCEmax のどちらかで効いていればOK

# 何個以上の全ターゲットで同じ向きの差が出ていれば「一貫している」とみなすか
min_total_targets_same_sign <- 2


# データ読み込み -----------------------------
diff_tbl <- read_csv(file_diff, show_col_types = FALSE)
# 列: target, feature, mean_is_S_FALSE, mean_is_S_TRUE, diff_S_minus_nonS


# 1) まず diff の絶対値が大きい行だけに絞る --------------------------
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)


# 2) 重要ターゲットでの効き具合を集計 -------------------------------
imp_effect <- diff_large %>%
  filter(target %in% important_targets) %>%
  group_by(feature) %>%
  summarise(
    n_imp_targets = n(),                                 # Jsc / PCEmax で効いている数
    max_abs_diff_imp = max(abs_diff),                   # その中で一番大きい差
    .groups = "drop"
  ) %>%
  filter(n_imp_targets >= min_imp_targets)


# 3) 全ターゲットでの「符号の一貫性」を評価 -------------------------
sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets = n(),                                    # 有効なターゲット数
    n_pos     = sum(sign > 0),
    n_neg     = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups   = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),                   # 多い方の符号の個数
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )


# 4) 重要ターゲットで効いていて、かつ符号も一貫している変数だけ抽出 ----
suspicious_vars <- imp_effect %>%
  inner_join(sign_consistency, by = "feature") %>%
  arrange(desc(max_abs_diff_imp))

suspicious_vars


Warning message:
"There was 1 warning in `summarise()`.
i In argument: `max_abs_diff_imp = max(abs_diff)`.
Caused by warning in `max()`:
! no non-missing arguments to max; returning -Inf"


feature,n_imp_targets,max_abs_diff_imp,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>,<int>,<chr>


In [50]:
library(dplyr)
library(readr)

# しきい値などの設定 --------------------------
file_diff <- "summary_feature_mean_diff_S_minus_nonS_by_target.csv"

# 「重要そうな目的変数」
important_targets <- c("FF")

# diff の絶対値しきい値
abs_diff_thr <- 0.01

# 何個以上の重要ターゲットで差が出ていれば採用するか
min_imp_targets <- 1   # 例: 少なくとも Jsc または PCEmax のどちらかで効いていればOK

# 何個以上の全ターゲットで同じ向きの差が出ていれば「一貫している」とみなすか
min_total_targets_same_sign <- 2


# データ読み込み -----------------------------
diff_tbl <- read_csv(file_diff, show_col_types = FALSE)
# 列: target, feature, mean_is_S_FALSE, mean_is_S_TRUE, diff_S_minus_nonS


# 1) まず diff の絶対値が大きい行だけに絞る --------------------------
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)


# 2) 重要ターゲットでの効き具合を集計 -------------------------------
imp_effect <- diff_large %>%
  filter(target %in% important_targets) %>%
  group_by(feature) %>%
  summarise(
    n_imp_targets = n(),                                 # Jsc / PCEmax で効いている数
    max_abs_diff_imp = max(abs_diff),                   # その中で一番大きい差
    .groups = "drop"
  ) %>%
  filter(n_imp_targets >= min_imp_targets)


# 3) 全ターゲットでの「符号の一貫性」を評価 -------------------------
sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets = n(),                                    # 有効なターゲット数
    n_pos     = sum(sign > 0),
    n_neg     = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups   = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),                   # 多い方の符号の個数
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )


# 4) 重要ターゲットで効いていて、かつ符号も一貫している変数だけ抽出 ----
suspicious_vars <- imp_effect %>%
  inner_join(sign_consistency, by = "feature") %>%
  arrange(desc(max_abs_diff_imp))

suspicious_vars


feature,n_imp_targets,max_abs_diff_imp,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>,<int>,<chr>
TAtime1,1,0.04395091,3,2,1,0.11394941,2,positive
Aarea,1,0.04288601,3,1,2,0.07347052,2,negative
Lay2thickness,1,0.02674466,2,0,2,0.03428392,2,negative


In [47]:
library(dplyr)
library(readr)
min_imp_targets <- 1
# 1) 入力ファイル
file_diff <- "summary_feature_mean_diff_S_minus_nonS_by_target.csv"

# 2) ターゲットとしきい値の設定 --------------------------
important_targets <- c("Voc")  # Voc だけを見る
abs_diff_thr <- 0.00000000000001           # |S-非S| が 0.05 以上の特徴量だけを怪しいとみなす

# 3) データ読み込み ---------------------------------------
diff_tbl <- read_csv(file_diff, show_col_types = FALSE)
# 列: target, feature, mean_is_S_FALSE, mean_is_S_TRUE, diff_S_minus_nonS

# 4) Voc だけに絞って、「差の大きい順」に並べる ----------------
suspicious_vars_voc <- diff_tbl %>%
  filter(target %in% important_targets) %>%     # Voc 行だけ
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>% # 絶対値を計算
  filter(abs_diff >= abs_diff_thr) %>%          # 閾値以上だけ残す
  arrange(desc(abs_diff))                       # 差が大きい順に並べる

suspicious_vars_voc


target,feature,mean_is_S_FALSE,mean_is_S_TRUE,diff_S_minus_nonS,abs_diff
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>


In [48]:
exists("oof_err")
ls()


[1] FALSE

[1] "abs_diff_thr"                "abs_error_thr"              
 [3] "abs_error_thr_list"          "abs_summary"                
 [5] "abs_summary_list"            "base_path"                  
 [7] "basic_list"                  "basic_summary"              
 [9] "bundle_folder_name"          "col_abserr"                 
[11] "col_id"                      "col_pred"                   
[13] "col_true"                    "diff_large"                 
[15] "diff_tbl"                    "f"                          
[17] "feat_diff"                   "feat_long"                  
[19] "feat_summary"                "feat_summary_all"           
[21] "feat_summary_list"           "file_diff"                  
[23] "file_feat_diff"              "file_feat_summary"          
[25] "imp_effect"                  "important_targets"          
[27] "key_features"                "min_imp_targets"            
[29] "min_total_targets_same_sign" "model_difficulty"           
[31] "oof_pce"                     "output_prefix"              
[33] "process_one_target"          "sign_consistency"           
[35] "src_dir"                     "src_files"                  
[37] "src_path"                    "summary_dir"                
[39] "summary_folder_name"         "suspicious_vars"            
[41] "suspicious_vars_allTargets"  "suspicious_vars_voc"        
[43] "target_config"               "targets"                    
[45] "test_oof"                    "tg"                         
[47] "tgt_dir"                     "top_k"                      
[49] "use_abs_thr_mode"            "use_top_k_mode"

In [13]:
test_oof <- readr::read_csv(
  file.path(
    base_path,
    bundle_folder_name,
    "PCE",
    "m_all_OH_rebuilt_PCEmax_pred_OOF_withError.csv"
  ),
  show_col_types = FALSE
)
names(test_oof)


[1] "SampleID"           "foldID"             "Observed"          
[4] "Predicted"          "Error"              "AbsError"          
[7] "Type"               "HighErrorFlag"      "HighErrorThreshold"

In [41]:
library(dplyr)
library(readr)
library(tidyr)

base_path     <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
output_prefix <- "outlier_results"

# 解析するターゲット
targets <- c("PCEmax", "Jsc", "Voc", "FF")

# 1) 各ターゲットについて、全数値列の S/非S 平均・SD を計算
feat_summary_list <- lapply(targets, function(tg) {
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))

  dat <- read_csv(file_all, show_col_types = FALSE)

  # 数値列を抽出（AbsError, is_S, ID系 など除外したい列はここで除く）
  num_cols <- names(dat)[sapply(dat, is.numeric)]
  num_cols <- setdiff(num_cols, c("AbsError", "is_S", "SampleID", "foldID"))

  if (length(num_cols) == 0) return(NULL)

  dat %>%
    group_by(is_S) %>%
    summarise(
      across(
        all_of(num_cols),
        list(mean = ~mean(.x, na.rm = TRUE),
             sd   = ~sd(.x,   na.rm = TRUE)),
        .names = "{.col}_{.fn}"
      ),
      .groups = "drop"
    ) %>%
    mutate(target = tg) %>%
    select(target, is_S, everything())
})

feat_summary_all <- bind_rows(feat_summary_list)

# 保存
file_feat_summary <- file.path(base_path, "summary_features_S_vs_nonS_allTargets_allNumeric.csv")
write_csv(feat_summary_all, file_feat_summary)


In [42]:
# 1) wide → long に展開して、mean だけ取り出す
feat_long <- feat_summary_all %>%
  pivot_longer(
    cols = -c(target, is_S),
    names_to  = "feat_stat",
    values_to = "value"
  ) %>%
  separate(feat_stat, into = c("feature", "stat"), sep = "_(?=[^_]+$)")

# 2) mean だけ残して、S/非S を横持ちにして差分
feat_diff <- feat_long %>%
  filter(stat == "mean") %>%
  select(target, is_S, feature, value) %>%
  pivot_wider(
    names_from  = is_S,
    values_from = value,
    names_prefix = "mean_is_S_"
  ) %>%
  mutate(
    diff_S_minus_nonS = mean_is_S_TRUE - mean_is_S_FALSE
  )

file_feat_diff <- file.path(base_path, "summary_feature_mean_diff_S_minus_nonS_allTargets_allNumeric.csv")
write_csv(feat_diff, file_feat_diff)


In [46]:
abs_diff_thr <- 0.01
min_total_targets_same_sign <- 1  # 何目的変数で同じ向きなら「一貫」とみなすか

# diff_tbl は summary_feature_mean_diff_... を読んだものとする
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)

sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets        = n(),
    n_pos            = sum(sign > 0),
    n_neg            = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )

suspicious_vars_allTargets <- sign_consistency %>%
  arrange(desc(max_abs_diff_all))

suspicious_vars_allTargets


feature,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<int>,<int>,<dbl>,<int>,<chr>
EgDFTp1M,2,0,2,0.11452591,2,negative
TAtime1,3,2,1,0.11394941,2,positive
Aarea,3,1,2,0.07347052,2,negative
Lay2thickness,2,0,2,0.03428392,2,negative


In [ ]:
# ここまでがあなたのコード
abs_diff_thr <- 0.05
min_total_targets_same_sign <- 2

diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)

sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets        = n(),
    n_pos            = sum(sign > 0),
    n_neg            = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )

suspicious_vars_allTargets <- sign_consistency %>%
  arrange(desc(max_abs_diff_all))

# ここから出力部分 ------------------------------

library(readr)

# ベースパス（必要ならあなたの環境に合わせて変更）
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"

# しきい値をファイル名に埋め込む（. を _ に置き換えると安全）
abs_str  <- gsub("\\.", "p", as.character(abs_diff_thr))
nsame_str <- as.character(min_total_targets_same_sign)

file_out <- file.path(
  base_path,
  paste0(
    "suspicious_features_allTargets_abs", abs_str,
    "_nsame", nsame_str,
    ".csv"
  )
)

write_csv(suspicious_vars_allTargets, file_out)

cat("書き出し先: ", file_out, "\n")


In [53]:
library(dplyr)
library(readr)
library(tidyr)

base_path     <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
output_prefix <- "outlier_results"

# 解析するターゲット
targets <- c("PCEmax", "Jsc", "Voc", "FF")

# 1) 各ターゲットについて、全数値列の S/非S 平均・SD を計算
feat_summary_list <- lapply(targets, function(tg) {
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))

  dat <- read_csv(file_all, show_col_types = FALSE)

  # 数値列を抽出（AbsError, is_S, ID系 など除外したい列はここで除く）
  num_cols <- names(dat)[sapply(dat, is.numeric)]
  num_cols <- setdiff(num_cols, c("AbsError", "is_S", "SampleID", "foldID"))

  if (length(num_cols) == 0) return(NULL)

  dat %>%
    group_by(is_S) %>%
    summarise(
      across(
        all_of(num_cols),
        list(mean = ~mean(.x, na.rm = TRUE),
             sd   = ~sd(.x,   na.rm = TRUE)),
        .names = "{.col}_{.fn}"
      ),
      .groups = "drop"
    ) %>%
    mutate(target = tg) %>%
    select(target, is_S, everything())
})

feat_summary_all <- bind_rows(feat_summary_list)

# 保存
file_feat_summary <- file.path(base_path, "summary_features_S_vs_nonS_allTargets_allNumeric.csv")
write_csv(feat_summary_all, file_feat_summary)


In [54]:
# 1) wide → long に展開して、mean だけ取り出す
feat_long <- feat_summary_all %>%
  pivot_longer(
    cols = -c(target, is_S),
    names_to  = "feat_stat",
    values_to = "value"
  ) %>%
  separate(feat_stat, into = c("feature", "stat"), sep = "_(?=[^_]+$)")

# 2) mean だけ残して、S/非S を横持ちにして差分
feat_diff <- feat_long %>%
  filter(stat == "mean") %>%
  select(target, is_S, feature, value) %>%
  pivot_wider(
    names_from  = is_S,
    values_from = value,
    names_prefix = "mean_is_S_"
  ) %>%
  mutate(
    diff_S_minus_nonS = mean_is_S_TRUE - mean_is_S_FALSE
  )

file_feat_diff <- file.path(base_path, "summary_feature_mean_diff_S_minus_nonS_allTargets_allNumeric.csv")
write_csv(feat_diff, file_feat_diff)


In [55]:
# 設定 ---------------------------
important_targets <- c("Jsc", "PCEmax", "Voc", "FF")  # 重視する目的変数
abs_diff_thr <- 0.05                    # |S-非S| のしきい値
min_imp_targets <- 1                    # 重要ターゲットの中で何個以上効いていれば採用するか
min_total_targets_same_sign <- 2        # 何目的変数で同じ向きなら「一貫」とみなすか

diff_tbl <- read_csv(file_feat_diff, show_col_types = FALSE)

# 1) 絶対差が大きい行だけ
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)

# 2) 重要ターゲットでの効き具合
imp_effect <- diff_large %>%
  filter(target %in% important_targets) %>%
  group_by(feature) %>%
  summarise(
    n_imp_targets   = n(),
    max_abs_diff_imp = max(abs_diff),
    .groups = "drop"
  ) %>%
  filter(n_imp_targets >= min_imp_targets)

# 3) 全ターゲットでの符号の一貫性
sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets       = n(),
    n_pos           = sum(sign > 0),
    n_neg           = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )

# 4) 両方の条件を満たす「怪しい変数」
suspicious_vars <- imp_effect %>%
  inner_join(sign_consistency, by = "feature") %>%
  arrange(desc(max_abs_diff_imp))

file_suspicious <- file.path(base_path, "suspicious_features_allNumeric_Jsc_PCEmax.csv")
write_csv(suspicious_vars, file_suspicious)

suspicious_vars


feature,n_imp_targets,max_abs_diff_imp,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>,<int>,<chr>
Error,2,1.00195386,4,4,0,1.00195386,4,positive
Predicted,2,0.85151568,3,2,1,0.85151568,2,positive
Observed,2,0.70514874,4,0,4,0.70514874,4,negative
UVIntP,2,0.43323695,2,0,2,0.43323695,2,negative
SMILESsnamenM_PC61BM,2,0.30211055,2,0,2,0.30211055,2,negative
Lay5electronodes1_LiF,2,0.27839196,3,2,1,0.27839196,2,positive
Lay2Mname_PEDOTPSS,1,0.26693467,2,2,0,0.26693467,2,positive
TAtimelast,2,0.26413965,3,1,2,0.26413965,2,negative
namessolvent1_CF,2,0.26150754,3,2,1,0.29753695,2,positive


In [56]:
# 設定 ---------------------------
important_targets <- c("Jsc", "PCEmax", "Voc", "FF")  # 重視する目的変数
abs_diff_thr <- 0.05                    # |S-非S| のしきい値
min_imp_targets <- 1                    # 重要ターゲットの中で何個以上効いていれば採用するか
min_total_targets_same_sign <- 2        # 何目的変数で同じ向きなら「一貫」とみなすか

diff_tbl <- read_csv(file_feat_diff, show_col_types = FALSE)

# 1) 絶対差が大きい行だけ
diff_large <- diff_tbl %>%
  mutate(abs_diff = abs(diff_S_minus_nonS)) %>%
  filter(abs_diff >= abs_diff_thr)

# 2) 重要ターゲットでの効き具合
imp_effect <- diff_large %>%
  filter(target %in% important_targets) %>%
  group_by(feature) %>%
  summarise(
    n_imp_targets   = n(),
    max_abs_diff_imp = max(abs_diff),
    .groups = "drop"
  ) %>%
  filter(n_imp_targets >= min_imp_targets)

# 3) 全ターゲットでの符号の一貫性
sign_consistency <- diff_large %>%
  mutate(sign = sign(diff_S_minus_nonS)) %>%
  group_by(feature) %>%
  summarise(
    n_targets       = n(),
    n_pos           = sum(sign > 0),
    n_neg           = sum(sign < 0),
    max_abs_diff_all = max(abs_diff),
    .groups = "drop"
  ) %>%
  mutate(
    n_same_sign = pmax(n_pos, n_neg),
    sign_major  = dplyr::case_when(
      n_pos > n_neg ~ "positive",
      n_neg > n_pos ~ "negative",
      TRUE          ~ "mixed"
    )
  ) %>%
  filter(
    sign_major != "mixed",
    n_same_sign >= min_total_targets_same_sign
  )

# 4) 両方の条件を満たす「怪しい変数」
suspicious_vars <- imp_effect %>%
  inner_join(sign_consistency, by = "feature") %>%
  arrange(desc(max_abs_diff_imp))

file_suspicious <- file.path(base_path, "suspicious_features_allNumeric_Jsc_PCEmax.csv")
write_csv(suspicious_vars, file_suspicious)

suspicious_vars


feature,n_imp_targets,max_abs_diff_imp,n_targets,n_pos,n_neg,max_abs_diff_all,n_same_sign,sign_major
<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>,<int>,<chr>
Error,4,1.0019539,4,4,0,1.0019539,4,positive
Predicted,3,0.8515157,3,2,1,0.8515157,2,positive
Observed,4,0.7051487,4,0,4,0.7051487,4,negative
UVIntP,2,0.4332369,2,0,2,0.4332369,2,negative
X149.4,2,0.4266010,2,0,2,0.4266010,2,negative
X141.4,2,0.4246305,2,0,2,0.4246305,2,negative
X108.4,2,0.4128079,2,0,2,0.4128079,2,negative
X101.4,2,0.4091954,2,0,2,0.4091954,2,negative
X150.4,2,0.3769882,2,0,2,0.3769882,2,negative


In [57]:
library(dplyr)
library(readr)
library(ggplot2)
library(purrr)

base_path     <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
output_prefix <- "outlier_results"

targets <- c("PCEmax", "Jsc", "Voc", "FF")

# 注目する変数（このリストにさっきの suspicious な変数を追加していけばOK）
focus_vars <- c("UVIntP", "TAtimelast", "Aarea", "EgDFTp1M")

# 「Error が大きいサンプル」の定義
# ここでは各ターゲットごとに AbsError の上位 5% を「Error大」とする例。
top_error_frac <- 0.05


In [61]:
analyze_target <- function(tg) {
  cat("\n====================\n")
  cat("Target:", tg, "\n")
  cat("====================\n\n")
  
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))
  
  dat <- read_csv(file_all, show_col_types = FALSE)
  
  # 必要な列があるかざっと確認
  needed <- c("Observed", "Predicted", "AbsError", "is_S")
  miss   <- setdiff(needed, names(dat))
  if (length(miss) > 0) {
    cat("[WARN]", tg, ": 必要な列が欠けています:", paste(miss, collapse = ", "), "\n")
    return(invisible(NULL))
  }
  
  # Error 大サンプルの閾値（上位 top_error_frac）
  thr_err <- quantile(dat$AbsError, probs = 1 - top_error_frac, na.rm = TRUE)
  dat <- dat %>%
    mutate(
      is_high_error = AbsError >= thr_err
    )
  
  cat("AbsError しきい値 (上位", top_error_frac*100, "%): ", thr_err, "\n", sep = "")
  cat("  高誤差サンプル数:", sum(dat$is_high_error, na.rm = TRUE),
      "/", nrow(dat), "\n")
  cat("  S (is_S=TRUE)   数:", sum(dat$is_S, na.rm = TRUE), "\n\n")
  
  # 注目変数が存在するものだけを対象に
  vars_exist <- intersect(focus_vars, names(dat))
  if (length(vars_exist) == 0) {
    cat("[INFO]", tg, ": 注目変数が1つも存在しません。\n")
  }
  
  # 各変数について、S/非S と高誤差/非高誤差の要約をコンソールに出す
  for (v in vars_exist) {
    cat("---- 変数:", v, " ----\n")
    tmp <- dat %>%
      group_by(is_S, is_high_error) %>%
      summarise(
        n    = n(),
        mean = mean(.data[[v]], na.rm = TRUE),
        sd   = sd(.data[[v]],   na.rm = TRUE),
        .groups = "drop"
      )
    print(tmp)
    cat("\n")
  }
  
  # プロット出力用フォルダ
  plot_dir <- file.path(out_dir, "plots_focusVars")
  if (!dir.exists(plot_dir)) dir.create(plot_dir, recursive = TRUE)
  
  # 1) 注目変数のヒストグラム（S/非S + 高誤差色分け）
  for (v in vars_exist) {
    p_hist <- ggplot(dat, aes(x = .data[[v]], fill = interaction(is_S, is_high_error))) +
      geom_histogram(position = "identity", alpha = 0.5, bins = 30, color = "black") +
      theme_bw() +
      labs(
        title = paste0(tg, " - Histogram of ", v),
        fill  = "is_S.is_high_error"
      )
    ggsave(
      filename = file.path(plot_dir, paste0("hist_", tg, "_", v, ".png")),
      plot     = p_hist, width = 6, height = 4
    )
    
    # 箱ひげ（S/非S × 高誤差）
    p_box <- ggplot(dat, aes(x = interaction(is_S, is_high_error), y = .data[[v]], fill = is_S)) +
      geom_boxplot(outlier.alpha = 0.5) +
      theme_bw() +
      labs(
        title = paste0(tg, " - Boxplot of ", v),
        x     = "is_S.is_high_error"
      )
    ggsave(
      filename = file.path(plot_dir, paste0("box_", tg, "_", v, ".png")),
      plot     = p_box, width = 5, height = 4
    )
  }
  
  # 2) Obs vs Pred 図上での位置（S/非S + 高誤差で色分け）
  dat <- dat %>%
    mutate(
      group_flag = case_when(
        is_S & is_high_error  ~ "S & highErr",
        is_S & !is_high_error ~ "S & normalErr",
        !is_S & is_high_error ~ "nonS & highErr",
        TRUE                  ~ "nonS & normalErr"
      )
    )
  
  p_scatter <- ggplot(dat,
                      aes(x = Observed, y = Predicted, color = group_flag)) +
    geom_point(alpha = 0.7) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
    theme_bw() +
    labs(
      title = paste0(tg, " - Obs vs Pred (colored by S/highError)"),
      color = "group"
    )
  
  ggsave(
    filename = file.path(plot_dir, paste0("obs_vs_pred_", tg, "_S_highErr.png")),
    plot     = p_scatter, width = 6, height = 5
  )
  
  cat("プロット出力先:", plot_dir, "\n\n")
  
  invisible(NULL)
}
walk(targets, analyze_target)



Target: PCEmax 

AbsError <U+3057><U+304D><U+3044><U+5024> (<U+4E0A><U+4F4D>5%): 2.581999
  <U+9AD8><U+8AA4><U+5DEE><U+30B5><U+30F3><U+30D7><U+30EB><U+6570>: 53 / 1045 
  S (is_S=TRUE)   <U+6570>: 18 

---- <U+5909><U+6570>: UVIntP  ----
# A tibble: 3 x 5
  is_S  is_high_error     n   mean    sd
  <lgl> <lgl>         <int>  <dbl> <dbl>
1 FALSE FALSE           992 -0.357 0.711
2 FALSE TRUE             35 -0.592 0.605
3 TRUE  TRUE             18 -0.798 0.473

---- <U+5909><U+6570>: TAtimelast  ----
# A tibble: 3 x 5
  is_S  is_high_error     n    mean    sd
  <lgl> <lgl>         <int>   <dbl> <dbl>
1 FALSE FALSE           992  0.124  0.932
2 FALSE TRUE             35  0.0580 0.948
3 TRUE  TRUE             18 -0.110  0.851

---- <U+5909><U+6570>: Aarea  ----
# A tibble: 3 x 5
  is_S  is_high_error     n   mean    sd
  <lgl> <lgl>         <int>  <dbl> <dbl>
1 FALSE FALSE           992 -0.790 0.363
2 FALSE TRUE             35 -0.784 0.322
3 TRUE  TRUE             18 -0.863 0.107

---- <U+5

In [62]:
library(dplyr)
library(readr)
library(ggplot2)
library(purrr)
library(tidyr)

# 設定 -----------------------------
base_path     <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
output_prefix <- "outlier_results"

target <- "Jsc"  # ここを PCEmax / Voc / FF に変えれば他の目的変数も見られる

# 注目する説明変数
focus_vars <- c(
  "UVIntP", "TAtimelast", "Aarea", "EgDFTp1M",   # 例: 怪しいとわかっているもの
  "TAtime1"                                      # 必要に応じて追加
)

# データ読み込み -------------------
out_dir  <- file.path(base_path, paste0(output_prefix, "_", target))
file_all <- file.path(out_dir, paste0("all_with_Sflag_", target, ".csv"))

dat <- read_csv(file_all, show_col_types = FALSE)

# 実際に存在する列だけに絞る
focus_vars <- intersect(focus_vars, names(dat))

if (length(focus_vars) == 0) {
  stop("指定した focus_vars がこの目的変数のデータに存在しません。")
}

# 目的変数・誤差など
y_var      <- "Observed"
yhat_var   <- "Predicted"
error_var  <- "AbsError"

# 「誤差が大きいサンプル」フラグ（上位5%）
top_error_frac <- 0.05
thr_err <- quantile(dat[[error_var]], probs = 1 - top_error_frac, na.rm = TRUE)

dat <- dat %>%
  mutate(
    is_high_error = !!as.name(error_var) >= thr_err
  )

cat("Target:", target, "\n")
cat("  高誤差しきい値 AbsError >=", thr_err, "\n")
cat("  高誤差サンプル数:", sum(dat$is_high_error, na.rm = TRUE), "/", nrow(dat), "\n\n")


Target: Jsc 
  <U+9AD8><U+8AA4><U+5DEE><U+3057><U+304D><U+3044><U+5024> AbsError >= 4.878994 
  <U+9AD8><U+8AA4><U+5DEE><U+30B5><U+30F3><U+30D7><U+30EB><U+6570>: 53 / 1045 



In [63]:
# long 形式にして facet_wrap でまとめて描く
plot_df <- dat %>%
  select(all_of(c(y_var, focus_vars, "is_S", "is_high_error"))) %>%
  pivot_longer(cols = all_of(focus_vars),
               names_to = "feature",
               values_to = "x_val")

# S / 非S × 高誤差の組み合わせを色で区別
plot_df <- plot_df %>%
  mutate(
    group_flag = case_when(
      is_S & is_high_error  ~ "S & highErr",
      is_S & !is_high_error ~ "S & normalErr",
      !is_S & is_high_error ~ "nonS & highErr",
      TRUE                  ~ "nonS & normalErr"
    )
  )

p_scatter_grid <- ggplot(plot_df,
                         aes(x = x_val, y = .data[[y_var]], color = group_flag)) +
  geom_point(alpha = 0.6, size = 1.2) +
  theme_bw() +
  facet_wrap(~ feature, scales = "free_x") +
  labs(
    title = paste0("Target = ", target,
                   " : ", y_var, " vs multiple features"),
    x     = "feature value",
    y     = y_var,
    color = "group"
  )

ggsave(
  filename = file.path(out_dir,
                       paste0("multiFeature_vs_", y_var, "_", target, ".png")),
  plot     = p_scatter_grid,
  width    = 10,
  height   = 8
)


In [64]:
# 箱ひげ: 目的変数ごとの高誤差 vs 非高誤差での分布差
p_box_grid <- plot_df %>%
  mutate(err_flag = ifelse(is_high_error, "highErr", "normalErr")) %>%
  ggplot(aes(x = err_flag, y = x_val, fill = err_flag)) +
  geom_boxplot(outlier.alpha = 0.5) +
  theme_bw() +
  facet_wrap(~ feature, scales = "free_y") +
  labs(
    title = paste0("Target = ", target,
                   " : feature distributions (highErr vs normal)"),
    x = "error group",
    y = "feature value"
  )

ggsave(
  filename = file.path(out_dir,
                       paste0("multiFeature_box_highErr_vs_normal_", target, ".png")),
  plot     = p_box_grid,
  width    = 10,
  height   = 8
)


In [65]:
cat("=== ", target, " : 高誤差 vs 非高誤差 の要約 ===\n")

for (v in focus_vars) {
  cat("\n--- feature:", v, " ---\n")
  tmp <- dat %>%
    mutate(err_flag = ifelse(is_high_error, "highErr", "normalErr")) %>%
    group_by(err_flag) %>%
    summarise(
      n    = n(),
      mean = mean(.data[[v]], na.rm = TRUE),
      sd   = sd(.data[[v]],   na.rm = TRUE),
      q1   = quantile(.data[[v]], 0.25, na.rm = TRUE),
      med  = median(.data[[v]], na.rm = TRUE),
      q3   = quantile(.data[[v]], 0.75, na.rm = TRUE),
      .groups = "drop"
    )
  print(tmp)
}


===  Jsc  : <U+9AD8><U+8AA4><U+5DEE> vs <U+975E><U+9AD8><U+8AA4><U+5DEE> <U+306E><U+8981><U+7D04> ===

--- feature: UVIntP  ---
# A tibble: 2 x 7
  err_flag      n   mean    sd    q1   med      q3
  <chr>     <int>  <dbl> <dbl> <dbl> <dbl>   <dbl>
1 highErr      53 -0.601 0.558    -1    -1 -0.0909
2 normalErr   992 -0.360 0.713    -1    -1  0.364 

--- feature: TAtimelast  ---
# A tibble: 2 x 7
  err_flag      n  mean    sd     q1   med    q3
  <chr>     <int> <dbl> <dbl>  <dbl> <dbl> <dbl>
1 highErr      53 0.331 0.885 -0.895     1     1
2 normalErr   992 0.106 0.932 -0.895     1     1

--- feature: Aarea  ---
# A tibble: 2 x 7
  err_flag      n   mean    sd     q1    med     q3
  <chr>     <int>  <dbl> <dbl>  <dbl>  <dbl>  <dbl>
1 highErr      53 -0.858 0.103 -0.969 -0.865 -0.744
2 normalErr   992 -0.787 0.367 -0.969 -0.846 -0.723

--- feature: EgDFTp1M  ---
# A tibble: 2 x 7
  err_flag      n   mean    sd     q1   med    q3
  <chr>     <int>  <dbl> <dbl>  <dbl> <dbl> <dbl>
1 highErr

In [66]:
library(dplyr)
library(readr)
library(purrr)

# ===== 設定 ======================================================

base_path     <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
output_prefix <- "outlier_results"

# 対象とする目的変数
targets <- c("PCEmax", "Jsc", "Voc", "FF")

# 誤差列名（あなたの環境に合わせて）
error_var <- "AbsError"

# 外れサンプルとみなす割合（上位何%の誤差を外れとするか）
top_error_frac <- 0.05


# ===== 1目的変数ぶんの処理をする関数 ============================

analyze_target_outlier_causes <- function(tg) {
  cat("\n=====================================\n")
  cat("Target:", tg, "\n")
  cat("=====================================\n")
  
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))
  
  if (!file.exists(file_all)) {
    cat("[WARN]", file_all, "が見つからないためスキップします。\n")
    return(invisible(NULL))
  }
  
  dat <- read_csv(file_all, show_col_types = FALSE)
  
  if (!error_var %in% names(dat)) {
    cat("[WARN]", tg, ": 列", error_var, "がありません。スキップします。\n")
    return(invisible(NULL))
  }
  
  # --- 1) 外れサンプルを特定 -----------------------------------
  thr_err <- quantile(dat[[error_var]], probs = 1 - top_error_frac, na.rm = TRUE)
  
  dat <- dat %>%
    mutate(
      is_outlier = .data[[error_var]] >= thr_err
    )
  
  n_out  <- sum(dat$is_outlier, na.rm = TRUE)
  n_all  <- nrow(dat)
  
  cat("  外れしきい値:", error_var, ">=", thr_err, "\n")
  cat("  外れサンプル数:", n_out, "/", n_all, "\n")
  
  if (n_out == 0 || n_out == n_all) {
    cat("  [INFO] 全部 or 全く外れになっているため、原因分析をスキップします。\n")
    return(invisible(NULL))
  }
  
  # --- 2) 説明変数の平均乖離を計算 -----------------------------
  # 説明変数として扱わない列
  exclude_cols <- c(
    "Observed", "Predicted", error_var,
    "is_outlier", "is_S",      # フラグ類
    "SampleID", "foldID"      # ID系があれば
  )
  
  num_cols <- names(dat)[sapply(dat, is.numeric)]
  feature_cols <- setdiff(num_cols, exclude_cols)
  
  if (length(feature_cols) == 0) {
    cat("  [INFO] 説明変数として扱える数値列がありません。\n")
    return(invisible(NULL))
  }
  
  cause_tbl <- map_dfr(
    feature_cols,
    function(v) {
      tmp <- dat %>%
        group_by(is_outlier) %>%
        summarise(
          n    = n(),
          mean = mean(.data[[v]], na.rm = TRUE),
          sd   = sd(.data[[v]],   na.rm = TRUE),
          .groups = "drop"
        )
      
      row_out  <- tmp[tmp$is_outlier == TRUE, , drop = FALSE]
      row_norm <- tmp[tmp$is_outlier == FALSE, , drop = FALSE]
      
      if (nrow(row_out) == 0 || nrow(row_norm) == 0) return(NULL)
      
      tibble(
        feature         = v,
        n_outlier       = row_out$n,
        n_normal        = row_norm$n,
        mean_outlier    = row_out$mean,
        mean_normal     = row_norm$mean,
        sd_outlier      = row_out$sd,
        sd_normal       = row_norm$sd,
        diff_mean       = mean_outlier - mean_normal,
        abs_diff_mean   = abs(diff_mean)
      )
    }
  )
  
  if (nrow(cause_tbl) == 0) {
    cat("  [INFO] 有効な要約が作れませんでした（NAが多いなど）。\n")
    return(invisible(NULL))
  }
  
  cause_tbl <- cause_tbl %>%
    arrange(desc(abs_diff_mean))
  
  cat("  上位5つの原因候補変数:\n")
  print(head(cause_tbl, 5))
  
  # --- 3) 結果を書き出し --------------------------------------
  file_cause <- file.path(
    out_dir,
    paste0("outlier_feature_divergence_", tg, ".csv")
  )
  
  write_csv(cause_tbl, file_cause)
  cat("  書き出し:", file_cause, "\n")
  
  invisible(NULL)
}


# ===== 4目的変数をまとめて処理 ==================================

walk(targets, analyze_target_outlier_causes)



Target: PCEmax 
  <U+5916><U+308C><U+3057><U+304D><U+3044><U+5024>: AbsError >= 2.581999 
  <U+5916><U+308C><U+30B5><U+30F3><U+30D7><U+30EB><U+6570>: 53 / 1045 
  <U+4E0A><U+4F4D>5<U+3064><U+306E><U+539F><U+56E0><U+5019><U+88DC><U+5909><U+6570>:
# A tibble: 5 x 9
  feature       n_outlier n_normal mean_outlier mean_normal sd_outlier sd_normal
  <chr>             <int>    <int>        <dbl>       <dbl>      <dbl>     <dbl>
1 SMILESsnamen~        53      992       0.358       0.0423      0.942     1.00 
2 Lay6electron~        53      992       0.0189      0.331       1.01      0.944
3 UVIntP               53      992      -0.662      -0.357       0.568     0.711
4 SMILESsnamen~        53      992      -0.472      -0.177       0.890     0.985
5 Lay6electron~        53      992      -0.547      -0.800       0.845     0.600
# i 2 more variables: diff_mean <dbl>, abs_diff_mean <dbl>
  <U+66F8><U+304D><U+51FA><U+3057>: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outli

In [67]:
library(dplyr)
library(readr)
library(ggplot2)
library(purrr)
library(tidyr)

base_path     <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
output_prefix <- "outlier_results"

targets    <- c("PCEmax", "Jsc", "Voc", "FF")
error_var  <- "AbsError"      # 誤差列
top_k_feat <- 5               # 各目的変数で上位いくつの原因候補変数を見るか
top_error_frac <- 0.05        # highErr の定義（AbsError 上位5%）


analyze_target_plots <- function(tg) {
  cat("\n==============================\n")
  cat("Target:", tg, "\n")
  cat("==============================\n")
  
  out_dir  <- file.path(base_path, paste0(output_prefix, "_", tg))
  file_all <- file.path(out_dir, paste0("all_with_Sflag_", tg, ".csv"))
  file_cause <- file.path(out_dir, paste0("outlier_feature_divergence_", tg, ".csv"))
  
  if (!file.exists(file_all)) {
    cat("[WARN]", file_all, "がありません。スキップ。\n")
    return(invisible(NULL))
  }
  if (!file.exists(file_cause)) {
    cat("[WARN]", file_cause, "がありません（原因候補ランキングが未作成）。スキップ。\n")
    return(invisible(NULL))
  }
  
  dat <- read_csv(file_all, show_col_types = FALSE)
  cause_tbl <- read_csv(file_cause, show_col_types = FALSE)
  
  # highErr フラグを付与（安全のため再計算）
  if (!error_var %in% names(dat)) {
    cat("[WARN]", tg, ": 列", error_var, "がありません。スキップ。\n")
    return(invisible(NULL))
  }
  
  thr_err <- quantile(dat[[error_var]], probs = 1 - top_error_frac, na.rm = TRUE)
  dat <- dat %>%
    mutate(
      is_high_error = .data[[error_var]] >= thr_err
    )
  
  cat("  highErr しきい値:", error_var, ">=", thr_err, "\n")
  cat("  highErr サンプル数:", sum(dat$is_high_error, na.rm = TRUE),
      "/", nrow(dat), "\n")
  
  # 上位 top_k_feat の原因候補変数を取得
  top_feats <- cause_tbl %>%
    arrange(desc(abs_diff_mean)) %>%
    slice_head(n = top_k_feat) %>%
    pull(feature)
  
  # 実際に存在しない列が入っていたら除く
  top_feats <- intersect(top_feats, names(dat))
  if (length(top_feats) == 0) {
    cat("  [INFO] 上位候補変数がデータに存在しません。\n")
    return(invisible(NULL))
  }
  
  cat("  プロット対象の上位候補変数:\n")
  print(top_feats)
  
  # プロット出力先
  plot_dir <- file.path(out_dir, "plots_outlier_causes_top")
  if (!dir.exists(plot_dir)) dir.create(plot_dir, recursive = TRUE)
  
  # 目的変数・予測値
  if (!all(c("Observed", "Predicted") %in% names(dat))) {
    cat("  [WARN] Observed / Predicted 列がないため Obs vs Pred 図はスキップします。\n")
  }
  
  # long 形式に変換
  plot_df <- dat %>%
    select(all_of(c("Observed", "Predicted", "is_high_error", "is_S", top_feats))) %>%
    pivot_longer(cols = all_of(top_feats),
                 names_to  = "feature",
                 values_to = "x_val") %>%
    mutate(
      group_flag = case_when(
        is_S & is_high_error  ~ "S & highErr",
        is_S & !is_high_error ~ "S & normalErr",
        !is_S & is_high_error ~ "nonS & highErr",
        TRUE                  ~ "nonS & normalErr"
      ),
      err_flag = ifelse(is_high_error, "highErr", "normalErr")
    )
  
  # 1) highErr vs normal の箱ひげ・ヒストグラム -----------------
  
  # 箱ひげ図（highErr vs normal）
  p_box <- ggplot(plot_df, aes(x = err_flag, y = x_val, fill = err_flag)) +
    geom_boxplot(outlier.alpha = 0.4) +
    theme_bw() +
    facet_wrap(~ feature, scales = "free_y") +
    labs(
      title = paste0("Target = ", tg,
                     " : feature distributions (highErr vs normal)"),
      x = "error group",
      y = "feature value"
    )
  
  ggsave(
    filename = file.path(plot_dir, paste0("box_highErr_vs_normal_", tg, ".png")),
    plot     = p_box,
    width    = 10, height = 8
  )
  
  # ヒストグラム（highErr / normal を色分け）
  p_hist <- ggplot(plot_df,
                   aes(x = x_val, fill = err_flag)) +
    geom_histogram(position = "identity", alpha = 0.5,
                   bins = 30, color = "black") +
    theme_bw() +
    facet_wrap(~ feature, scales = "free_x") +
    labs(
      title = paste0("Target = ", tg,
                     " : histograms of top features (highErr vs normal)"),
      x = "feature value",
      y = "count",
      fill = "error group"
    )
  
  ggsave(
    filename = file.path(plot_dir, paste0("hist_highErr_vs_normal_", tg, ".png")),
    plot     = p_hist,
    width    = 10, height = 8
  )
  
  # 2) Obs vs Pred における highErr 点の分布 --------------------
  if (all(c("Observed", "Predicted") %in% names(dat))) {
    p_scatter <- ggplot(dat,
                        aes(x = Observed, y = Predicted,
                            color = is_high_error, shape = is_S)) +
      geom_point(alpha = 0.7) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
      theme_bw() +
      scale_color_manual(values = c("FALSE" = "gray50", "TRUE" = "red")) +
      labs(
        title = paste0("Target = ", tg,
                       " : Obs vs Pred (highErr in red)"),
        color = "is_high_error",
        shape = "is_S"
      )
    
    ggsave(
      filename = file.path(plot_dir, paste0("obs_vs_pred_highErr_", tg, ".png")),
      plot     = p_scatter,
      width    = 7, height = 6
    )
  }
  
  cat("  プロット出力先:", plot_dir, "\n")
  invisible(NULL)
}

# 4目的変数まとめて実行
walk(targets, analyze_target_plots)



Target: PCEmax 
  highErr <U+3057><U+304D><U+3044><U+5024>: AbsError >= 2.581999 
  highErr <U+30B5><U+30F3><U+30D7><U+30EB><U+6570>: 53 / 1045 
  <U+30D7><U+30ED><U+30C3><U+30C8><U+5BFE><U+8C61><U+306E><U+4E0A><U+4F4D><U+5019><U+88DC><U+5909><U+6570>:
[1] "SMILESsnamenM_PC71BM" "Lay6electronodes2_Al" "UVIntP"              
[4] "SMILESsnamenM_PC61BM" "Lay6electronodes2_Ag"
  <U+30D7><U+30ED><U+30C3><U+30C8><U+51FA><U+529B><U+5148>: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier/outlier_results_PCEmax/plots_outlier_causes_top 

Target: Jsc 
  highErr <U+3057><U+304D><U+3044><U+5024>: AbsError >= 4.878994 
  highErr <U+30B5><U+30F3><U+30D7><U+30EB><U+6570>: 53 / 1045 
  <U+30D7><U+30ED><U+30C3><U+30C8><U+5BFE><U+8C61><U+306E><U+4E0A><U+4F4D><U+5019><U+88DC><U+5909><U+6570>:
[1] "Error"                 "namessolvent1_oDCB"    "Lay2Mname_PEDOTPSS"   
[4] "Lay5electronodes1_LiF" "SMILESsnamenM_PC61BM" 
  <U+30D7><U+30ED><U+30C3><U+30C8><U+51FA><U+529B><U+5148>: /Volume